In [1]:
import numpy as np



In [2]:
import sys
sys.path.append('../')

from pqcqec.simulate.jax_statevector import (
    run_circuit_with_state,
    run_many_states,
    create_zero_state,
    create_ones_state,
    build_jax_circuit,
)

build_numba_circuit = build_jax_circuit  # For compatibility with noise module

from pqcqec.circuits.generate import generate_random_circuit
from pqcqec.circuits.pqc_circuits import list_LEL_ZZ

from pqcqec.noise.builder import add_noise_to_base_ops

In [3]:
NUM_QUBITS = 2
NUM_GATES = 4
NUM_GATE_BLOCKS = 2

# Generate a random circuit
circuit_ops = generate_random_circuit(NUM_QUBITS, NUM_GATES, seed=50, backend='list')
for op in circuit_ops:
    print(op)

('h', [1], [])
('z', [1], [])
('cx', [0, 1], [])
('z', [0], [])


In [4]:
ideal_numba_ops = build_numba_circuit(circuit_ops)
printable_ideal_ops = list(zip(*ideal_numba_ops))
for op in printable_ideal_ops:
    print(np.array(op)) 

[ 3.  1. -1.  0.]
[ 2.  1. -1.  0.]
[7. 0. 1. 0.]
[ 2.  0. -1.  0.]


In [5]:

x_noise = np.ones((NUM_GATES,), dtype=np.float32) * 0.1
z_noise = np.ones((NUM_GATES,), dtype=np.float32) * 0.1

noisy_circuit_ops = add_noise_to_base_ops(circuit_ops, x_noise, z_noise)

for op in noisy_circuit_ops:
    print(op)
    

('h', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('z', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('cx', [0, 1], [])
('rx', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.1)])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('z', [0], [])
('rx', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.1)])


In [6]:
noisy_numba_ops = build_numba_circuit(noisy_circuit_ops)

printable_noisy_numba_ops = list(zip(*noisy_numba_ops))
for op in printable_noisy_numba_ops:
    print(np.array(op)) 

[ 3.  1. -1.  0.]
[ 4.   1.  -1.   0.1]
[ 6.   1.  -1.   0.1]
[ 2.  1. -1.  0.]
[ 4.   1.  -1.   0.1]
[ 6.   1.  -1.   0.1]
[7. 0. 1. 0.]
[ 4.   0.  -1.   0.1]
[ 6.   0.  -1.   0.1]
[ 4.   1.  -1.   0.1]
[ 6.   1.  -1.   0.1]
[ 2.  0. -1.  0.]
[ 4.   0.  -1.   0.1]
[ 6.   0.  -1.   0.1]


In [7]:
gate_ids, wire1s, wire2s, params = ideal_numba_ops


In [8]:
initial_state = create_zero_state(NUM_QUBITS)
final_state = run_circuit_with_state(initial_state, NUM_QUBITS, gate_ids, wire1s, wire2s, params)
print("Final state from ideal circuit:")
print(final_state)
# run_many_states(NUM_QUBITS, gate_ids, wire1s, wire2s, params) 

Final state from ideal circuit:
[ 0.70710677+0.j -0.70710677+0.j  0.        -0.j -0.        +0.j]


In [9]:
initial_state = create_ones_state(NUM_QUBITS)
final_state = run_circuit_with_state(initial_state, NUM_QUBITS, gate_ids, wire1s, wire2s, params)
print("Final state from ideal circuit:")
print(final_state)
# run_many_states(NUM_QUBITS, gate_ids, wire1s, wire2s, params) 

Final state from ideal circuit:
[ 0.        +0.j -0.        +0.j -0.70710677+0.j -0.70710677+0.j]


In [10]:
from pqcqec.circuits.pqc_circuits import list_LEL_ZZ

pqc_circuit_ops = []
pre_params = np.zeros((NUM_QUBITS,3), dtype=np.float32)
post_params = np.zeros((NUM_QUBITS,3), dtype=np.float32)
theta_zz = np.zeros((NUM_QUBITS,), dtype=np.float32)

op_ctr = 0
for i, op in enumerate(circuit_ops):

    pqc_circuit_ops.append(op)
    for q in op[1]:
        pqc_circuit_ops.append(('rx', [q], [x_noise[i]]))  # Identity gate as a placeholder for noise
        pqc_circuit_ops.append(('rz', [q], [z_noise[i]]))  # Identity gate as a placeholder for noise

    if (i + 1) % NUM_GATE_BLOCKS == 0:
        pqc_circuit_ops += list_LEL_ZZ(NUM_QUBITS, pre_params, theta_zz, post_params)



In [11]:
for op in pqc_circuit_ops:
    print(op)

('h', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('z', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('rz', [0], [np.float32(0.0)])
('rx', [0], [np.float32(0.0)])
('rz', [0], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('rx', [1], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('cnot', [0, 1], [])
('rz', [1], [np.float32(0.0)])
('cnot', [0, 1], [])
('cnot', [1, 0], [])
('rz', [0], [np.float32(0.0)])
('cnot', [1, 0], [])
('rz', [0], [np.float32(0.0)])
('rx', [0], [np.float32(0.0)])
('rz', [0], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('rx', [1], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('cx', [0, 1], [])
('rx', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.1)])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('z', [0], [])
('rx', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.0)])
('rx', [0], [np.float32(0.0)])
('rz', [0], [np.float32(0.0)])
('rz', [1], [np

# Testing Circuit Template Approach

Now let's test the template-based approach for efficient circuit generation.

In [12]:
from pqcqec.circuits.templates import build_pqc_circuit_template

# Build the template once (this is the "compilation" step)
template = build_pqc_circuit_template(
    base_ops=circuit_ops,
    num_qubits=NUM_QUBITS,
    num_gate_blocks=NUM_GATE_BLOCKS,
    add_noise=True,
    add_pqc_layers=True
)

print(f"Template created: {template}")
print(f"Total gates in template: {len(template)}")

Template created: CircuitTemplate(num_gates=50, param_sources={None, 'z_noise', 'theta_zz', 'post_params', 'x_noise', 'pre_params'})
Total gates in template: 50


## Parameter Structure for Template

The template expects parameters in specific shapes:
- **base**: 1D array `(num_gates,)` - parameters for base circuit gates
- **x_noise**: 1D array `(num_gates,)` - X noise per gate
- **z_noise**: 1D array `(num_gates,)` - Z noise per gate  
- **pre_params**: 3D array `(num_pqc_layers, num_qubits, 3)` - pre-local unitaries (RZ-RX-RZ)
- **theta_zz**: 1D array `(num_qubits,)` - ZZ entangling angles (shared across layers)
- **post_params**: 3D array `(num_pqc_layers, num_qubits, 3)` - post-local unitaries (RZ-RX-RZ)

Where `num_pqc_layers = num_gates // num_gate_blocks`

In [13]:
# Now instantiate the template with actual parameters (fast!)
# Note: pre_params and post_params need to be 3D: (num_pqc_layers, num_qubits, 3)
# theta_zz remains 1D: (num_qubits,) - shared across all PQC layers
# Calculate number of PQC layers based on how many blocks we have
num_pqc_layers = len(circuit_ops) // NUM_GATE_BLOCKS

# Reshape parameters to 3D for pre/post params
pre_params_3d = np.zeros((num_pqc_layers, NUM_QUBITS, 3), dtype=np.float32)
post_params_3d = np.zeros((num_pqc_layers, NUM_QUBITS, 3), dtype=np.float32)

# If we have 2D arrays, broadcast them to all layers
if pre_params.ndim == 2:
    pre_params_3d[:] = pre_params[np.newaxis, :, :]
else:
    pre_params_3d = pre_params

if post_params.ndim == 2:
    post_params_3d[:] = post_params[np.newaxis, :, :]
else:
    post_params_3d = post_params

# theta_zz stays 1D
param_dict = {
    'base': np.array([op[2][0] if len(op[2]) > 0 else 0.0 for op in circuit_ops], dtype=np.float32),
    'x_noise': x_noise,
    'z_noise': z_noise,
    'pre_params': pre_params_3d,
    'theta_zz': theta_zz,  # Keep 1D
    'post_params': post_params_3d
}

print(f"Number of PQC layers: {num_pqc_layers}")
print(f"Parameter shapes:")
print(f"  base: {param_dict['base'].shape}")
print(f"  x_noise: {param_dict['x_noise'].shape}")
print(f"  z_noise: {param_dict['z_noise'].shape}")
print(f"  pre_params: {param_dict['pre_params'].shape} (should be ({num_pqc_layers}, {NUM_QUBITS}, 3))")
print(f"  theta_zz: {param_dict['theta_zz'].shape} (should be ({NUM_QUBITS},))")
print(f"  post_params: {param_dict['post_params'].shape} (should be ({num_pqc_layers}, {NUM_QUBITS}, 3))")
print()

instantiated_circuit = template.instantiate(param_dict)

print("Instantiated circuit operations:")
for op in instantiated_circuit:
    print(op)

Number of PQC layers: 2
Parameter shapes:
  base: (4,)
  x_noise: (4,)
  z_noise: (4,)
  pre_params: (2, 2, 3) (should be (2, 2, 3))
  theta_zz: (2,) (should be (2,))
  post_params: (2, 2, 3) (should be (2, 2, 3))

Instantiated circuit operations:
('h', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('z', [1], [])
('rx', [1], [np.float32(0.1)])
('rz', [1], [np.float32(0.1)])
('rz', [0], [np.float32(0.0)])
('rx', [0], [np.float32(0.0)])
('rz', [0], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('rx', [1], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('cnot', [0, 1], [])
('rz', [1], [np.float32(0.0)])
('cnot', [0, 1], [])
('cnot', [1, 0], [])
('rz', [0], [np.float32(0.0)])
('cnot', [1, 0], [])
('rz', [0], [np.float32(0.0)])
('rx', [0], [np.float32(0.0)])
('rz', [0], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('rx', [1], [np.float32(0.0)])
('rz', [1], [np.float32(0.0)])
('cx', [0, 1], [])
('rx', [0], [np.float32(0.1)])
('rz', [0], [np.float32(0.1)])

In [14]:
# Verify that template produces the same result as the iterative approach
print("Comparing old approach vs template approach:")
print(f"Old approach length: {len(pqc_circuit_ops)}")
print(f"Template approach length: {len(instantiated_circuit)}")
print(f"Match: {len(pqc_circuit_ops) == len(instantiated_circuit)}")

# Check if operations match
all_match = True
for i, (old_op, new_op) in enumerate(zip(pqc_circuit_ops, instantiated_circuit)):
    if old_op != new_op:
        print(f"Mismatch at index {i}:")
        print(f"  Old: {old_op}")
        print(f"  New: {new_op}")
        all_match = False

if all_match:
    print("✓ All operations match!")

Comparing old approach vs template approach:
Old approach length: 50
Template approach length: 50
Match: True
✓ All operations match!


## Performance Comparison

Let's benchmark the performance difference between the iterative approach and the template approach.

In [15]:
import time

# Benchmark the old iterative approach
num_iterations = 1000

start = time.time()
for _ in range(num_iterations):
    pqc_circuit_ops_iter = []
    for i, op in enumerate(circuit_ops):
        pqc_circuit_ops_iter.append(op)
        for q in op[1]:
            pqc_circuit_ops_iter.append(('rx', [q], [x_noise[i]]))
            pqc_circuit_ops_iter.append(('rz', [q], [z_noise[i]]))
        if (i + 1) % NUM_GATE_BLOCKS == 0:
            pqc_circuit_ops_iter += list_LEL_ZZ(NUM_QUBITS, pre_params, theta_zz, post_params)
old_time = time.time() - start

# Benchmark the template approach
start = time.time()
for _ in range(num_iterations):
    instantiated_circuit_bench = template.instantiate(param_dict)
new_time = time.time() - start

print(f"Old iterative approach: {old_time:.4f} seconds for {num_iterations} iterations")
print(f"Template approach: {new_time:.4f} seconds for {num_iterations} iterations")
print(f"Speedup: {old_time / new_time:.2f}x faster")

Old iterative approach: 0.0064 seconds for 1000 iterations
Template approach: 0.0061 seconds for 1000 iterations
Speedup: 1.05x faster


## Using Template with Different Parameters

The key advantage: you can now reuse the template with different parameter values very efficiently!

In [16]:
# Example: Generate circuits with different noise levels
noise_levels = [0.01, 0.05, 0.1, 0.2]
circuits_with_different_noise = []

for noise_level in noise_levels:
    # Create new parameter dictionary with different noise
    # Note: pre_params and post_params must be 3D: (num_pqc_layers, num_qubits, 3)
    new_param_dict = {
        'base': np.array([op[2][0] if len(op[2]) > 0 else 0.0 for op in circuit_ops], dtype=np.float32),
        'x_noise': np.ones((NUM_GATES,), dtype=np.float32) * noise_level,
        'z_noise': np.ones((NUM_GATES,), dtype=np.float32) * noise_level,
        'pre_params': np.random.randn(num_pqc_layers, NUM_QUBITS, 3).astype(np.float32) * 0.1,
        'theta_zz': np.random.randn(NUM_QUBITS).astype(np.float32) * 0.1,
        'post_params': np.random.randn(num_pqc_layers, NUM_QUBITS, 3).astype(np.float32) * 0.1
    }
    
    # Instantiate template with new parameters
    circuit = template.instantiate(new_param_dict)
    circuits_with_different_noise.append((noise_level, circuit))
    
print(f"Generated {len(circuits_with_different_noise)} circuits with different noise levels")
for noise_level, circuit in circuits_with_different_noise:
    print(f"  Noise level {noise_level}: {len(circuit)} gates")

Generated 4 circuits with different noise levels
  Noise level 0.01: 50 gates
  Noise level 0.05: 50 gates
  Noise level 0.1: 50 gates
  Noise level 0.2: 50 gates
